In [ ]:

import json
import pennylane as qml
import pennylane.numpy as np

# Write any helper functions you need

def S_op(phi):
    """Returns the S(phi) phase operator matrix."""
    return np.array([[np.exp(1j * phi), 0],
                     [0, np.exp(-1j * phi)]])

def U_op(a):
    """Returns the U(a) unitary matrix."""
    # Ensure 1 - a**2 is non-negative for real sqrt
    sqrt_term = np.sqrt(np.clip(1 - a**2, 0, 1))
    return np.array([[a, sqrt_term],
                     [sqrt_term, -a]])

def get_qsp_matrix(angles, a):
    """
    Constructs the QSP circuit matrix for given phase angles and 'a' value.
    The circuit is S(phi_0) U(a) S(phi_1) U(a) S(phi_2) U(a) S(phi_3).
    
    Args:
        angles (np.array(float)): Array of four phase angles [phi_0, phi_1, phi_2, phi_3].
        a (float): The parameter 'a' for the U(a) gate.
        
    Returns:
        (np.array(complex)): The 2x2 unitary matrix representing the QSP circuit.
    """
    phi0, phi1, phi2, phi3 = angles
    
    # Build the product from right to left
    matrix = S_op(phi3)
    matrix = U_op(a) @ matrix
    matrix = S_op(phi2) @ matrix
    matrix = U_op(a) @ matrix
    matrix = S_op(phi1) @ matrix
    matrix = U_op(a) @ matrix
    matrix = S_op(phi0) @ matrix
    
    return matrix

def coefficients(angles):
    
    """ This function returns the coefficients associated with the polynomial generated by 
    the QSP routine as a function of the phase angles.
    
    Args:
        - angles (np.array(float)): Array of real numbers containing the four phase angles,
        in reverse order of application.
    
    Returns:       
        - (np.array(complex)): A numpy array containing the coefficients of the polynomial
        generated by QSP, where the first element is the coefficient of the cubic term and
        the second element is for the linear term.
    """
    
    # The polynomial is P(a) = alpha * a^3 + beta * a
    # We need to solve for alpha and beta using two distinct values of 'a'.
    # Choose two distinct 'a' values within [-1, 1] that are not 0 and where a1^2 != a2^2.
    a1 = 0.5
    a2 = 0.8
    
    # Calculate P(a1) and P(a2) by getting the top-left element of the QSP matrix
    M1 = get_qsp_matrix(angles, a1)
    P1 = M1[0, 0]
    
    M2 = get_qsp_matrix(angles, a2)
    P2 = M2[0, 0]
    
    # We have a system of linear equations:
    # P1 = alpha * a1^3 + beta * a1  => P1/a1 = alpha * a1^2 + beta
    # P2 = alpha * a2^3 + beta * a2  => P2/a2 = alpha * a2^2 + beta
    
    # Let y1 = P1/a1, y2 = P2/a2
    # Let x1 = a1^2, x2 = a2^2
    # y1 = alpha * x1 + beta
    # y2 = alpha * x2 + beta
    
    y1 = P1 / a1
    y2 = P2 / a2
    
    x1 = a1**2
    x2 = a2**2
    
    # Solve for alpha: (y1 - y2) = alpha * (x1 - x2)
    alpha = (y1 - y2) / (x1 - x2)
    
    # Solve for beta: beta = y1 - alpha * x1
    beta = y1 - alpha * x1
    
    return np.array([alpha, beta])


# These functions are responsible for testing the solution.


def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    coeffs = coefficients(ins)
    # Convert numpy complex numbers to a list of lists [real, imag] for JSON serialization
    outs = [[np.real(elem).numpy(), np.imag(elem).numpy()] for elem in coeffs]
    
    return str(outs)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.allclose(solution_output, expected_output, rtol = 1e-4)


# These are the public test cases
test_cases = [
    ('[0.5,0.8,1.0,1.0]', '[[-0.1707976, -2.4084889], [-0.8166822, 2.2507432]]'),
    ('[-0.20409113, -0.91173829, 0.91173829, 0.20409113]', '[[2.5,0],[-1.5,0]]')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")
